#### Image Patch Generator

In a Vision Transformer (ViT), **patch embedding** and **positional encoding** provide two different kinds of information:

- Patch embedding answers: **"What visual content is in this patch?"**
- Positional encoding answers: **"Where did this patch come from in the image?"**

#### Patch Embedding

An image is divided into fixed-size patches. For example, a $224 \times 224$ RGB image with $16 \times 16$ patches produces:

$$
N = \frac{224}{16} \times \frac{224}{16} = 14 \times 14 = 196
$$

patches.

Each patch initially has:

$$
16 \times 16 \times 3 = 768
$$

pixel values. A learned linear projection maps each flattened patch to a model embedding of dimension $D$:

$$
\mathbf{e}_i = \mathbf{x}_i \mathbf{W}_E + \mathbf{b}_E
$$

where:

- $\mathbf{x}_i$ is the flattened patch
- $\mathbf{W}_E$ is a learned projection matrix
- $\mathbf{e}_i \in \mathbb{R}^{D}$ is the patch embedding

In PyTorch, this is often implemented efficiently with a convolution:

```python
patch_embedding = nn.Conv2d(
    in_channels=3,
    out_channels=embedding_dim,
    kernel_size=patch_size,
    stride=patch_size,
)
```

The convolution extracts non-overlapping patches and projects each one into the transformer's embedding space.

In [6]:
import torch
import torch.nn as nn

class PatchEmbedding(nn.Module):
    def __init__(self, img_size = 224, patch_size=16, in_channels=3, embed_dim=768):
        super().__init__()
        self.patch_size = patch_size
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.proj(x)  # (Batch, embed_dim, H/patch_size, W/patch_size)
        x = x.flatten(2)  # (Batch, embed_dim, num_patches)
        x = x.transpose(1, 2)  # (Batch, num_patches, embed_dim)
        return x

# Test the patch embedding module on 2 random input image
patch_embedding = PatchEmbedding()
emb = patch_embedding.forward(torch.randn(2, 3, 224, 224))
print(emb)
print(emb.shape)

tensor([[[ 1.0580, -1.2604, -1.2151,  ..., -0.6381, -1.0716,  0.3156],
         [-0.7911, -0.2093, -0.5171,  ..., -1.0402,  0.3193,  0.2552],
         [ 0.2670,  0.1941,  0.4432,  ...,  0.6830, -1.1446,  0.1988],
         ...,
         [ 0.2477, -0.1686, -0.0929,  ...,  0.2854,  0.3638, -0.7861],
         [ 0.4972,  0.3854, -0.1355,  ...,  0.8408, -0.5085, -0.5447],
         [-0.0492, -0.2759, -1.0058,  ..., -0.2392, -0.4177,  0.5196]],

        [[-0.0823,  0.9640,  0.6048,  ...,  0.3369, -0.2952, -0.2790],
         [-0.1553,  1.0321, -0.2507,  ...,  0.4657,  0.4850, -1.9685],
         [ 0.8691,  0.0054,  0.0338,  ..., -0.2712, -0.0350, -0.9748],
         ...,
         [-1.0041, -0.6259, -0.6684,  ...,  0.3166,  0.3174,  0.1734],
         [-0.2212, -0.0398,  0.0103,  ...,  0.4134,  0.5216, -0.5918],
         [ 0.2098, -0.0037, -0.2942,  ..., -0.2298, -1.8524,  0.0444]]],
       grad_fn=<TransposeBackward0>)
torch.Size([2, 196, 768])


#### Positional Encoding

Self-attention does not inherently understand token order or spatial position. Without positional information, it treats the patch embeddings like an unordered set.

A positional vector is therefore added to every patch embedding:

$$
\mathbf{z}_i = \mathbf{e}_i + \mathbf{p}_i
$$

where:

- $\mathbf{e}_i$ describes the patch's visual content
- $\mathbf{p}_i$ describes its location
- $\mathbf{z}_i$ contains both content and position

For learned positional embeddings:

```python
position_embedding = nn.Parameter(
    torch.randn(1, num_patches + 1, embedding_dim)
)
```

The extra position is usually for the `[CLS]` token.

In [9]:
class PositionalEncoding(nn.Module):
    def __init__(self, embed_dim, max_len=5000):
        super().__init__()
        self.pos_embedding = nn.Parameter(torch.zeros(1, max_len+1, embed_dim))

    def forward(self, x):
        x = x + self.pos_embedding[:, :x.size(1), :]
        return x

position_encoding = PositionalEncoding(embed_dim=768)
output = position_encoding.forward(torch.randn(2, 196, 768))
print(output)
print(output.shape)

tensor([[[ 0.0293, -0.1964,  0.3883,  ..., -0.3639, -2.0639,  1.3130],
         [ 0.5872, -0.6621,  0.9056,  ..., -0.2303, -0.2008, -0.3447],
         [-0.2912,  0.1886,  0.9864,  ..., -1.2744, -0.7912,  0.4923],
         ...,
         [-0.9221, -1.1735,  1.4046,  ..., -1.1884, -0.7861,  1.2260],
         [ 0.3343,  0.4235, -1.4888,  ...,  0.4311, -2.0389, -0.5199],
         [-0.3345,  0.0171,  0.6617,  ..., -1.6152, -0.8239, -0.3477]],

        [[-0.1730, -0.1917,  0.2440,  ...,  1.0526,  0.1115,  1.4806],
         [ 1.2659, -0.3111,  0.3192,  ...,  0.5134, -1.1315,  0.7498],
         [ 0.6946, -1.0140, -1.7278,  ..., -1.3158, -0.3372, -1.5579],
         ...,
         [-0.9044,  0.4978, -1.0838,  ..., -0.0046, -0.0766,  0.6288],
         [-0.9640, -0.4236,  0.5410,  ...,  0.9219,  1.4269,  0.1143],
         [ 0.4684, -0.0323,  1.3354,  ..., -0.3306, -0.7844,  0.1483]]],
       grad_fn=<AddBackward0>)
torch.Size([2, 196, 768])


#### Intuitive Comparison

A useful intuition is:

- **Patch embedding is a feature extractor.** The convolution divides the image into patches and converts the pixels in each patch into a feature vector. It tells the transformer **what is in each patch**.
- **Positional encoding is a spatial address or coordinate label.** It adds a different learned vector to each patch token to tell the transformer **where that patch came from** in the image.

For example, the patch embedding may detect features resembling an eye and a mouth. Positional encoding lets the model distinguish an eye above a mouth from a mouth above an eye. It is like attaching row-and-column coordinates to every patch before giving the patches to the transformer.

The two vectors have the same embedding dimension but different jobs:

| Component | Intuitive role | Information supplied |
|---|---|---|
| Patch embedding | Convolutional feature extraction | What appears in the patch |
| Positional encoding | Spatial address / coordinate label | Where the patch belongs |

Both are necessary because self-attention can compare patch content globally, but by itself it has no built-in knowledge of token order or the image's 2D layout. Without patch embeddings there are no visual features to analyze; without positional encodings the transformer sees those features as an unordered collection of patches.

## Patch Embedding vs. a Full Convolutional Network Stack

Patch embedding does **not always work better** than a full convolutional feature extractor with pooling. It is mainly better suited to the **Transformer architecture and its token-based input format**.

### What Patch Embedding Does

A ViT patch embedding is itself a convolution:

```python
nn.Conv2d(
    in_channels=3,
    out_channels=embed_dim,
    kernel_size=patch_size,
    stride=patch_size,
  )
```

With `kernel_size=16` and `stride=16`, it:

1. Splits the image into non-overlapping $16 \times 16$ patches.
2. Projects each patch into an embedding vector.
3. Produces a sequence of tokens for the Transformer.

For a $224 \times 224$ image:

$$
224 \times 224 \longrightarrow 14 \times 14 \longrightarrow 196\text{ tokens}
$$

Its purpose is primarily **tokenization and projection**, not sophisticated feature extraction.

### Full Convolutional Network

A conventional CNN applies several operations:

```text
Convolution -> Activation -> Convolution -> Pooling -> ...
```

This builds a hierarchy:

- Early layers detect edges and textures.
- Middle layers detect shapes and parts.
- Deeper layers detect objects and semantic concepts.

Pooling gradually reduces spatial resolution while preserving important local features.

### Why ViTs Use Simple Patch Embedding

#### 1. Transformers need tokens

A Transformer expects a sequence shaped approximately like:

$$
(\text{batch}, \text{number of tokens}, \text{embedding dimension})
$$

Patch embedding converts the image directly into that format:

$$
(B,C,H,W) \rightarrow (B,N,D)
$$

A full CNN could also produce tokens, but it would perform much of the feature extraction before the Transformer receives the image.

#### 2. Self-attention performs the later feature extraction

In a CNN, stacked convolutions extract increasingly complex features. In a ViT, the Transformer blocks perform this job through self-attention and MLP layers.

The intended division of work is:

```text
Patch embedding -> create initial visual tokens
Transformer blocks -> learn relationships and higher-level features
```

A deep CNN before the Transformer may therefore duplicate some of the Transformer's role.

#### 3. Patch embedding preserves more raw information

Pooling is intentionally lossy. Max pooling retains only the maximum value from each local region:

$$
\begin{bmatrix}
1 & 3 \\
2 & 7
\end{bmatrix}
\longrightarrow 7
$$

Other values are discarded. This gives CNNs useful translation invariance, but exact spatial details can disappear.

Patch embedding learns a projection of all values in each patch rather than explicitly selecting only the maximum. However, patch embedding is still lossy when `embed_dim` is smaller than the flattened patch dimension.

#### 4. It allows global interactions early

CNNs initially combine information only from nearby pixels. Their receptive field expands gradually as layers are stacked.

After patch embedding, self-attention allows every patch to interact directly with every other patch:

$$
\operatorname{Attention}(Q,K,V) =
\operatorname{softmax}\left(\frac{QK^\top}{\sqrt{D}}\right)V
$$

A patch in the top-left corner can immediately attend to one in the bottom-right corner. This is useful when recognizing an object depends on relationships between distant regions.

#### 5. It imposes fewer assumptions

Convolutions assume that:

- Nearby pixels are especially related.
- The same local detector should be used everywhere.
- Local features should be composed hierarchically.

These are valuable **inductive biases**, especially when training data is limited.

A ViT imposes fewer of these assumptions and allows attention to learn relationships from data. With sufficiently large datasets and models, this flexibility can lead to better performance.

### When CNNs Can Be Better

CNNs often perform better when:

- The training dataset is small.
- Compute or memory is limited.
- Local textures are especially important.
- Strong translation invariance is useful.
- Efficient inference is required.

Self-attention has approximately quadratic complexity in the number of tokens:

$$
O(N^2)
$$

Reducing the patch size increases $N$ quickly. For a $224 \times 224$ image:

- $16 \times 16$ patches produce $196$ tokens.
- $8 \times 8$ patches produce $784$ tokens.
- $4 \times 4$ patches produce $3136$ tokens.

### Hybrid Approaches

Many modern vision models combine both ideas:

```text
Convolutional stem
        |
        v
Visual feature map
        |
        v
Flatten into tokens
        |
        v
Transformer blocks
```

A shallow convolutional stem can extract stable local features and reduce resolution before attention handles global relationships.

| Approach | Strength |
|---|---|
| Patch embedding | Simple tokenization, early global attention, and scalability with large datasets |
| Deep CNN with pooling | Strong local feature extraction and good data efficiency |
| CNN-Transformer hybrid | Combines local inductive bias with global attention |

The key point is that patch embedding is not inherently a better feature extractor than a full CNN. It is a **simpler interface between images and Transformers**, allowing the Transformer itself to learn most of the feature hierarchy.

#### Multihead Attention

In [11]:
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim, num_heads)

    def forward(self, x):
        return self.attn(x, x, x)


# Test the MultiHeadAttention module
embed_dim = 64
num_heads = 8
seq_length = 10
batch_size = 2

x = torch.rand(seq_length, batch_size, embed_dim)
mha = MultiHeadAttention(embed_dim, num_heads)
output, _ = mha(x)
print(output.shape)

torch.Size([10, 2, 64])
